## Step 1. Install the latest version of `hda`

You can run the next cell to install the latest version of `hda`:

In [ ]:
pip install hda -U

*__Note__: version used in this notebook is `2.17`*.

## Step 2. Import `hda` module

The HDA Client provides a fully compliant Python3 Client that can be used to search and download products using the Harmonized Data Access WEkEO API. First let's import the `hda` functions:

In [10]:
from hda import Client, Configuration

## Step 3. Configure credentials and load `hda` Client

### Method 1 (not regular users)

Pass your credentials directly in the script:

In [ ]:
# Configure your credentials without a .hdarc file
conf = Configuration(user = "xxx", password = "yyy")
hda_client = Client(config = conf)

## Step 4. Create the request and download data

### Get the dataset metadata

Here we are going to download the following Copernicus Land dataset: __EO:EEA:DAT:CLMS_HRVPP_VPP__.

To create our request we can ask to the API what parameters are needed.
To do so we use the `metadata()` function:

In [12]:
help(hda_client.metadata)

Help on method metadata in module hda.api:

metadata(dataset_id) method of hda.api.Client instance
    Returns the metadata object for the given dataset.
    
    :param dataset_id: The dataset ID
    :type dataset_id: str



In [13]:
# Request metadata of a dataset
hda_client.metadata(dataset_id="EO:EEA:DAT:HRL:TCF")

{'type': 'object',
 'title': 'Queryable',
 'properties': {'dataset_id': {'title': 'Dataset_id',
   'type': 'string',
   'oneOf': [{'const': 'EO:EEA:DAT:HRL:TCF',
     'title': 'EO:EEA:DAT:HRL:TCF',
     'group': None}]},
  'bbox': {'title': 'bbox',
   'type': 'array',
   'minItems': 4,
   'maxItems': 4,
   'items': [{'type': 'number', 'maximum': 180, 'minimum': -180},
    {'type': 'number', 'maximum': 90, 'minimum': -90},
    {'type': 'number', 'maximum': 180, 'minimum': -180},
    {'type': 'number', 'maximum': 90, 'minimum': -90}]},
  'productType': {'title': 'Product Type',
   'type': 'string',
   'oneOf': [{'const': 'Broadleaved Cover Density',
     'title': 'Broadleaved Cover Density',
     'group': None},
    {'const': 'Coniferous Cover Density',
     'title': 'Coniferous Cover Density',
     'group': None},
    {'const': 'Dominant Leaf Type',
     'title': 'Dominant Leaf Type',
     'group': None},
    {'const': 'Dominant Leaf Type Change',
     'title': 'Dominant Leaf Type Chang

## Create the request

Based on this information we can create the request below.

In [ ]:
# first attempt to download all the European data
query = {
  "dataset_id": "EO:EEA:DAT:HRL:TCF",
  "bbox": [
    22, 34, 35.7, 36.1
  ],
  "productType": "Tree Cover Density",
  "resolution": "100m",
  "year": "2023",
  "itemsPerPage": 200,
  "startIndex": 500
}

  # "bbox": [-22, 24.28417701, 32, 72.66440807], 
  # but xipre and some islands from greece were missing xd. Download the mising part of the data.


<div class="alert alert-block alert-info">
    📌 <b>Note</b>: the geographical coordinates in the <code>bbox</code> are ordered as: <code>[longitude_min, latitude_min, longitude_max, latitude_max]</code>
</div>

## Search data

The `search()` function launches the search of the data you requested with the specific parameters. It may take some time, as the server processes it.

In [15]:
matches = hda_client.search(query)
print(matches)

SearchResults[items=29,volume=4.9MB]


## Download file(s)

On WEkEO's JupyterHub you are limited to 20GB of stockage space, so be careful of the total size of files your request generated.  

### Download files in the current working directory

You can run `matches.download()` to download all the files of your request.  
Please [read the documentation](https://hda.readthedocs.io/en/latest/usage.html#advanced-client-usage) for advanced usage such as:
- downloading first result: `matches[0].download()`
- downloading last result: `matches[-1].download()`
- downloading first 10 results: `matches[:10].download()`
- downloading even results: `matches[::2].download()`
- etc.

For the purpose of this example, we are going to fetch the last result:

In [ ]:
OUTPUT_PATH = '../data/treecover2023'
matches.download(OUTPUT_PATH)

The `download()` function launches the download of the file(s) your request generated. They will be downloaded in the same folder as this notebook unless you specify an existing directory as `OUTPUT_PATH`.

In [16]:
import os

TOTAL_ITEMS = 760
ITEMS_PER_PAGE = 200
OUTPUT_PATH = '../data/treecover2023'

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

for start_index in range(0, TOTAL_ITEMS, ITEMS_PER_PAGE):
    print(f"Requesting items {start_index} to {start_index + ITEMS_PER_PAGE}...")
    
    query = {
        "dataset_id": "EO:EEA:DAT:HRL:TCF",
        "bbox": [22, 34, 35.7, 36.1],
        "productType": "Tree Cover Density",
        "resolution": "100m",
        "year": "2023",
        "itemsPerPage": ITEMS_PER_PAGE,
        "startIndex": start_index
    }
    
    matches = hda_client.search(query)
    
    # This downloads only the 200 items found in THIS specific search
    matches.download(OUTPUT_PATH)

print("All chunks downloaded successfully.")

Requesting items 0 to 200...


Requesting items 200 to 400...
Requesting items 400 to 600...
Requesting items 600 to 800...
All chunks downloaded successfully.


## Aggregate all files into one single raster

In [17]:
import os
import zipfile
import glob
import subprocess
from pathlib import Path
import rasterio
from rasterio.merge import merge

# 1. Setup paths
# Adjust OUTPUT_PATH to your environment
OUTPUT_PATH = '../data/treecover2023/' 
temp_extract_dir = os.path.join('../data/temp_tifs')
output_file = os.path.join(OUTPUT_PATH, 'europe_combined_treecover2023_complete.tif')

os.makedirs(temp_extract_dir, exist_ok=True)


In [18]:

# 2. Unzip all files
print("Extracting ZIP files...")
for zip_path in glob.glob(os.path.join(OUTPUT_PATH, "*.zip")):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Extract only .tif files to save space
        for member in zip_ref.namelist():
            if member.endswith('.tif'):
                zip_ref.extract(member, temp_extract_dir)

# 3. Find all extracted .tif files
tif_files = glob.glob(os.path.join(temp_extract_dir, "**/*.tif"), recursive=True)

if not tif_files:
    print("No .tif files found. Check your paths!")
    exit()

print(f"Found {len(tif_files)} tiles. Starting merge...")


Extracting ZIP files...
Found 770 tiles. Starting merge...


In [20]:
# 1. Open all your tif files
src_files_to_mosaic = [rasterio.open(f) for f in tif_files]

# 2. Merge them into one array
# 'method=max' ensures trees don't get covered by black borders
mosaic, out_trans = merge(src_files_to_mosaic, method='max')

# 3. Copy metadata from the first file and update
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "crs": src_files_to_mosaic[0].crs
})

# 4. Write the final merged file
output_path = os.path.join(temp_extract_dir, "europe_tree_cover_2023_complete.tif")
with rasterio.open(output_path, "w", **out_meta) as dest:
    dest.write(mosaic)